# 09 — Clustering no supervisado (K-Means, GMM, Aglomerativo y HDBSCAN)

**Pregunta:** ¿los agarres aparecen solos, sin usar las etiquetas?

Se le dan al algoritmo las filas **sin decirle qué agarre es cada una**, se le pide que las separe en grupos
por parecido, y al final se compara si esos grupos coinciden con los agarres reales.

El clustering se hace **directamente sobre las columnas del dataset** (sin PCA).

Las figuras **no se muestran en el notebook**: se guardan como PDF en `figures/clustering/<dataset>/`
(con la fecha de generación en la esquina), igual que en el notebook 07. Cada método genera su
tabla de contingencia (en **proporciones**, no conteos), su vista **2D** y su vista **3D**.

Cada sección de cálculo está implementada como **función** (`run_kmeans`, `run_gmm`, …); las celdas
solo las llaman y muestran resultados.

### Cómo reutilizarlo con otro dataset
Cambia `DATA_PATH` (y opcionalmente `FEATURE_COLS`) en la primera celda de código.
Si `FEATURE_COLS = None`, las columnas se detectan automáticamente: todas las numéricas
que no sean metadatos (`stimulus`, `subject`, `origin_db`, ventanas, etc.).

| Sección | Contenido |
|---|---|
| 1 | Carga (con detección automática de features) |
| 2 | K-Means con k = número de agarres: métricas, tabla % y vistas 2D/3D |
| 3 | K-Means con otros valores de k |
| 4 | GMM: barrido de n_components con BIC/AIC |
| 5 | Aglomerativo: comparación de linkages |
| 6 | HDBSCAN con min_samples explícito |
| 7 | ¿Es mejor que el azar? (permutación) |
| 8 | Resumen y guardado |
| 9 | Interpretación |

In [ ]:
from _provenance import Sello
sello = Sello('08_clustering.ipynb')

In [ ]:
# Cambia SOLO estas dos líneas para probar otro dataset.
DATA_PATH = '../data/prueba/df_flat_wnd_mean.csv'
FEATURE_COLS = None   # None = detecta automáticamente las columnas numéricas no-meta

In [ ]:
# ---------------- Configuración (normalmente no se toca) ----------------
LABEL_COL, SUBJECT_COL, DB_COL = 'stimulus', 'subject', 'origin_db'
META_COLS = {LABEL_COL, SUBJECT_COL, DB_COL, 'repetition', 'window',
             'window_index', 'window_size_samples', 'window_size_ms'}
REST = 0
K_RANGE = range(2, 13)                # valores de k a probar en la sección 3
GMM_N_RANGE = range(2, 13)            # n_components a probar en la sección 4
GMM_COVARIANCE = 'full'
AGG_LINKAGES = ['ward', 'complete', 'average']   # linkages a probar (sección 5)
HDB_MIN_SIZES = [15, 20, 30, 40]      # tamaños mínimos de grupo (sección 6)
HDB_MIN_SAMPLES = [1, 3, 5, 10]       # min_samples explícitos (sección 6)
N_PERM = 1000                         # repeticiones de la prueba de azar
RANDOM_STATE = 42
EMBED_2D = 'tsne'                     # cómo dibujar en 2D: 'tsne' | 'umap'
# -------------------------------------------------------------------------

import os
import warnings
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.cluster import KMeans, HDBSCAN, AgglomerativeClustering
from sklearn.manifold import TSNE
from sklearn.mixture import GaussianMixture
from sklearn.metrics import (adjusted_rand_score, normalized_mutual_info_score,
                             silhouette_score)

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

# Fecha de generación de las figuras (sello en la esquina de cada PDF)
GEN_DATE = datetime.now().strftime('%Y-%m-%d %H:%M')


def style_ax(ax):
    ax.set_facecolor('#FFFFFF')
    ax.spines[['top', 'right']].set_visible(False)
    ax.spines[['left', 'bottom']].set_color('#CCCCCC')
    ax.tick_params(colors='#444444', labelsize=8)


def savefig(fig, name):
    """Guarda la figura en PDF dentro de SAVE_DIR y libera la memoria (sin plt.show)."""
    fig.text(0.99, 0.01, f'Generado: {GEN_DATE}', ha='right', va='bottom',
             fontsize=7, color='#666666', transform=fig.transFigure)
    fig.savefig(f'{SAVE_DIR}/{name}.pdf', dpi=300, bbox_inches='tight')
    plt.close(fig)

In [ ]:
# ---------- Funciones de análisis (una por sección) ----------

def cargar_datos(data_path, feature_cols=None, rest=REST, meta_cols=META_COLS,
                 label_col=LABEL_COL, subject_col=SUBJECT_COL, db_col=DB_COL):
    """Lee el CSV, detecta features, quita el reposo y arma X/y/groups."""
    df = pd.read_csv(data_path)
    missing = [c for c in [label_col, subject_col, db_col] if c not in df.columns]
    if missing:
        raise KeyError(f'Faltan columnas en el CSV ({len(missing)}): {missing}')

    if feature_cols is None:
        feature_cols = [c for c in df.columns
                        if c not in meta_cols and pd.api.types.is_numeric_dtype(df[c])]
        if not feature_cols:
            raise KeyError('No se detectó ninguna columna numérica de features.')
    else:
        missing_f = [c for c in feature_cols if c not in df.columns]
        if missing_f:
            raise KeyError(f'feature_cols no está en el CSV ({len(missing_f)}): {missing_f[:10]}')

    df = df[df[label_col] != rest].copy().reset_index(drop=True)
    df['group'] = df[subject_col].astype(str) + '_' + df[db_col].astype(str)

    X = df[feature_cols].values.astype(float)
    y = df[label_col].values
    groups = df['group'].values
    classes = sorted(np.unique(y).tolist())
    return df, X, y, groups, classes


def clusters_per_subject(labels, groups):
    # ≈ número de agarres → agrupa agarres. ≈ 1 → agrupa personas.
    s = pd.DataFrame({'g': groups, 'c': labels})
    s = s[s['c'] >= 0]
    return s.groupby('g')['c'].nunique().mean()


def evaluate(labels, y, groups):
    return {
        'ARI_agarre': adjusted_rand_score(y, labels),
        'NMI_agarre': normalized_mutual_info_score(y, labels),
        'ARI_sujeto': adjusted_rand_score(groups, labels),
        'grupos_por_sujeto': clusters_per_subject(labels, groups),
    }


def run_kmeans(X, y, groups, k, n_init=50, seed=RANDOM_STATE):
    labels = KMeans(n_clusters=k, n_init=n_init, random_state=seed).fit_predict(X)
    res = {'k': k, 'silhouette': silhouette_score(X, labels), **evaluate(labels, y, groups)}
    return labels, res


def barrido_kmeans(X, y, ks=K_RANGE, n_init=20, seed=RANDOM_STATE):
    rows = []
    for k in ks:
        lab = KMeans(n_clusters=k, n_init=n_init, random_state=seed).fit_predict(X)
        rows.append({'k': k, 'silhouette': silhouette_score(X, lab),
                     'ARI_agarre': adjusted_rand_score(y, lab),
                     'NMI_agarre': normalized_mutual_info_score(y, lab)})
    df = pd.DataFrame(rows)
    best = int(df.loc[df['silhouette'].idxmax(), 'k'])
    return df, best


def run_gmm(X, y, groups, n_range=GMM_N_RANGE, covariance=GMM_COVARIANCE, seed=RANDOM_STATE):
    """Barrido de n_components con BIC/AIC + modelo final en el mejor BIC."""
    rows = []
    for n in n_range:
        g = GaussianMixture(n_components=n, covariance_type=covariance,
                            n_init=5, random_state=seed).fit(X)
        lab = g.predict(X)
        rows.append({'n_components': n, 'BIC': g.bic(X), 'AIC': g.aic(X),
                     'silhouette': silhouette_score(X, lab),
                     'ARI_agarre': adjusted_rand_score(y, lab),
                     'NMI_agarre': normalized_mutual_info_score(y, lab)})
    df = pd.DataFrame(rows)
    best_n = int(df.loc[df['BIC'].idxmin(), 'n_components'])

    g = GaussianMixture(n_components=best_n, covariance_type=covariance,
                        n_init=10, random_state=seed).fit(X)
    labels = g.predict(X)
    res = {'n_components': best_n, 'silhouette': silhouette_score(X, labels),
           **evaluate(labels, y, groups)}
    return df, best_n, labels, res


def run_aglomerativo(X, y, groups, k, linkages=AGG_LINKAGES):
    labels, rows = {}, []
    for link in linkages:
        lab = AgglomerativeClustering(n_clusters=k, linkage=link).fit_predict(X)
        labels[link] = lab
        rows.append({'linkage': link, 'silhouette': silhouette_score(X, lab),
                     **evaluate(lab, y, groups)})
    df = pd.DataFrame(rows).set_index('linkage')
    best = df['ARI_agarre'].idxmax()
    return df, labels, best


def run_hdbscan(X, y, groups, min_sizes=HDB_MIN_SIZES, min_samples=HDB_MIN_SAMPLES):
    """Grid min_cluster_size × min_samples (explícito); mejor combinación por ARI."""
    labels, rows = {}, []
    for mcs in min_sizes:
        for ms in min_samples:
            lab = HDBSCAN(min_cluster_size=mcs, min_samples=ms).fit_predict(X)
            labels[(mcs, ms)] = lab
            rows.append({'min_cluster_size': mcs, 'min_samples': ms,
                         'n_grupos': len(set(lab[lab >= 0])),
                         '%_ruido': 100 * (lab < 0).mean(),
                         **evaluate(lab, y, groups)})
    df = pd.DataFrame(rows).set_index(['min_cluster_size', 'min_samples'])
    valid = df[df['n_grupos'] > 1]
    best = valid['ARI_agarre'].idxmax() if len(valid) else None
    return df, labels, best


def prueba_azar(y, tests, n_perm=N_PERM, seed=RANDOM_STATE):
    """tests: [{'nombre', 'labels', 'ari', 'color'}]. Distribución nula por algoritmo."""
    rng = np.random.default_rng(seed)
    nulls, p_vals = {}, {}
    for t in tests:
        null = np.array([adjusted_rand_score(rng.permutation(y), t['labels'])
                         for _ in range(n_perm)])
        nulls[t['nombre']] = null
        p_vals[t['nombre']] = (np.sum(null >= t['ari']) + 1) / (n_perm + 1)
    return nulls, p_vals


def armar_asignaciones(df, etiquetas, label_col=LABEL_COL,
                       subject_col=SUBJECT_COL, db_col=DB_COL):
    """etiquetas: {nombre_columna: array de etiquetas} → DataFrame por fila."""
    out = df[[subject_col, db_col, 'group', label_col]].copy()
    for col, lab in etiquetas.items():
        out[col] = lab
    return out


def guardar_resultados(results_dir, resumen, asignaciones, barridos):
    """Escribe resumen.csv + asignaciones + barridos (índice solo si no es RangeIndex)."""
    os.makedirs(results_dir, exist_ok=True)
    asignaciones.to_csv(f'{results_dir}/asignaciones.csv', index=False)
    for nombre, tabla in barridos.items():
        con_indice = not isinstance(tabla.index, pd.RangeIndex)
        tabla.to_csv(f'{results_dir}/{nombre}.csv', index=con_indice)
    pd.DataFrame([resumen]).to_csv(f'{results_dir}/resumen.csv', index=False)

## 1. Carga

El dataset ya viene transformado y normalizado: solo se carga y se quita el reposo.
Si `FEATURE_COLS = None` se detectan automáticamente las columnas numéricas que no son metadatos.
El sujeto se identifica con el ID compuesto `subject + origin_db`.

In [ ]:
df, X, y, groups, CLASSES = cargar_datos(DATA_PATH, FEATURE_COLS)
N_CLASSES = len(CLASSES)
CLASS_COLORS = dict(zip(CLASSES, sns.color_palette('tab10', N_CLASSES)))

N_FEATURES = len(FEATURE_COLS) if FEATURE_COLS is not None else X.shape[1]
DATASET_TAG = f'{Path(DATA_PATH).stem}__{N_FEATURES}f'
SAVE_DIR = f'../figures/clustering/{DATASET_TAG}'
RESULTS_DIR = f'../results/clustering/{DATASET_TAG}'
os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

print(f'Dataset: {DATA_PATH} ({N_FEATURES} columnas)')
print(f'Figuras: {SAVE_DIR}\nResultados: {RESULTS_DIR}')
print(f'Filas: {len(df)} | agarres ({N_CLASSES}): {CLASSES} | sujetos: {len(np.unique(groups))}')

In [ ]:
# ---------- Funciones de gráficas (ven los datos como globals del dataset) ----------

def plot_contingency(labels, title, fname):
    # Filas = grupos encontrados; columnas = agarres reales.
    # Se muestra la PROPORCIÓN de cada agarre repartida entre grupos (columna = 100%).
    ct = pd.crosstab(pd.Series(labels, name='Grupo encontrado'),
                     pd.Series(y, name='Agarre real'))
    ct = ct.reindex(columns=CLASSES, fill_value=0)
    ct_norm = ct.div(ct.sum(axis=0), axis=1)
    ct_pct = (ct_norm * 100).round(0).astype(int)
    ct.to_csv(f'{RESULTS_DIR}/{fname}_conteos.csv')   # conteos crudos, por si hacen falta

    fig, ax = plt.subplots(figsize=(1.1 * N_CLASSES + 2, 0.55 * len(ct) + 1.8), dpi=300)
    fig.patch.set_facecolor('#F7F9FC')
    sns.heatmap(ct_norm, annot=ct_pct, fmt='d', cmap='Blues', vmin=0, vmax=1,
                linewidths=0.5, linecolor='#FFFFFF',
                cbar_kws={'label': 'Proporción del agarre (%)'}, ax=ax)
    ax.set_title(title, fontweight='bold', fontsize=10)
    plt.tight_layout()
    savefig(fig, fname)


def proyectar(X, n_components, embed=EMBED_2D, seed=RANDOM_STATE):
    """Embedding 2D o 3D (t-SNE o UMAP); se calcula una vez y se reutiliza."""
    if embed == 'umap':
        import umap
        coords = umap.UMAP(n_components=n_components, n_neighbors=30, min_dist=0.1,
                           random_state=seed).fit_transform(X)
        return coords, 'UMAP'
    coords = TSNE(n_components=n_components, perplexity=30, init='pca',
                  random_state=seed).fit_transform(X)
    return coords, 't-SNE'


def plot_clusters_2d(labels, title, fname):
    # Izquierda: grupos encontrados. Derecha: agarres reales. Mismas coordenadas.
    fig, axes = plt.subplots(1, 2, figsize=(15, 6.2), dpi=300, sharex=True, sharey=True)
    fig.patch.set_facecolor('#F7F9FC')

    ax = axes[0]
    found = [c for c in sorted(set(labels)) if c >= 0]
    pal = sns.color_palette('Set2', max(1, len(found)))
    if (labels < 0).any():
        m = labels < 0
        ax.scatter(E2D[m, 0], E2D[m, 1], s=14, color='#BDBDBD', alpha=0.5, label='Ruido (−1)')
    for c, col in zip(found, pal):
        m = labels == c
        ax.scatter(E2D[m, 0], E2D[m, 1], s=28, color=col, alpha=0.85,
                   edgecolor='white', linewidth=0.4, label=f'Grupo {c}')
        ax.text(np.median(E2D[m, 0]), np.median(E2D[m, 1]), str(c), fontsize=11,
                fontweight='bold', ha='center', va='center',
                bbox=dict(boxstyle='circle,pad=0.25', fc='white', ec=col, lw=1.5))
    ax.set_title('Grupos encontrados', fontweight='bold')

    ax = axes[1]
    for cls in CLASSES:
        m = y == cls
        ax.scatter(E2D[m, 0], E2D[m, 1], s=28, color=CLASS_COLORS[cls], alpha=0.85,
                   edgecolor='white', linewidth=0.4, label=f'Agarre {cls}')
    ax.set_title('Agarres reales', fontweight='bold')

    for ax in axes:
        style_ax(ax)
        ax.set_xlabel(f'{AXIS} 1', fontweight='bold')
        ax.legend(fontsize=7, loc='best', frameon=True)
    axes[0].set_ylabel(f'{AXIS} 2', fontweight='bold')
    fig.suptitle(title, fontweight='bold', fontsize=11)
    plt.tight_layout()
    savefig(fig, fname)


def plot_clusters_3d(labels, title, fname):
    """Misma vista espejo que plot_clusters_2d pero en 3D (PDF estático)."""
    fig = plt.figure(figsize=(15, 6.4), dpi=300)
    fig.patch.set_facecolor('#F7F9FC')

    ax = fig.add_subplot(1, 2, 1, projection='3d')
    found = [c for c in sorted(set(labels)) if c >= 0]
    pal = sns.color_palette('Set2', max(1, len(found)))
    if (labels < 0).any():
        m = labels < 0
        ax.scatter(E3D[m, 0], E3D[m, 1], E3D[m, 2], s=12, color='#BDBDBD',
                   alpha=0.5, label='Ruido (−1)')
    for c, col in zip(found, pal):
        m = labels == c
        ax.scatter(E3D[m, 0], E3D[m, 1], E3D[m, 2], s=26, color=col, alpha=0.85,
                   edgecolor='white', linewidth=0.3, label=f'Grupo {c}')
        ax.text(np.median(E3D[m, 0]), np.median(E3D[m, 1]), np.median(E3D[m, 2]),
                str(c), fontsize=10, fontweight='bold', ha='center', va='center',
                bbox=dict(boxstyle='circle,pad=0.2', fc='white', ec=col, lw=1.2))
    ax.set_title('Grupos encontrados', fontweight='bold')

    ax = fig.add_subplot(1, 2, 2, projection='3d')
    for cls in CLASSES:
        m = y == cls
        ax.scatter(E3D[m, 0], E3D[m, 1], E3D[m, 2], s=26, color=CLASS_COLORS[cls],
                   alpha=0.85, edgecolor='white', linewidth=0.3, label=f'Agarre {cls}')
    ax.set_title('Agarres reales', fontweight='bold')

    for ax in fig.axes:
        ax.view_init(elev=30, azim=-60)
        ax.set_xlabel(f'{AXIS} 1', fontweight='bold', fontsize=8)
        ax.set_ylabel(f'{AXIS} 2', fontweight='bold', fontsize=8)
        ax.set_zlabel(f'{AXIS} 3', fontweight='bold', fontsize=8)
        ax.legend(fontsize=6, loc='best', frameon=True)
    fig.suptitle(title, fontweight='bold', fontsize=11)
    fig.tight_layout()
    savefig(fig, fname)


def plot_barrido_k(sweep_df, best_k):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), dpi=300)
    fig.patch.set_facecolor('#F7F9FC')
    axes[0].plot(sweep_df['k'], sweep_df['silhouette'], marker='o', color='#1E88E5')
    axes[0].set_ylabel('Silhouette', fontweight='bold')
    axes[0].set_title('Separación de los grupos según k', fontweight='bold')
    axes[1].plot(sweep_df['k'], sweep_df['ARI_agarre'], marker='o', color='#F4511E', label='ARI')
    axes[1].plot(sweep_df['k'], sweep_df['NMI_agarre'], marker='s', color='#43A047', label='NMI')
    axes[1].set_ylabel('Coincidencia con agarres', fontweight='bold')
    axes[1].set_title('Coincidencia con los agarres según k', fontweight='bold')
    axes[1].legend(fontsize=8)
    for ax in axes:
        style_ax(ax)
        ax.axvline(N_CLASSES, color='#999999', linestyle=':', linewidth=1)
        ax.set_xlabel('k (número de grupos)', fontweight='bold')
    plt.tight_layout()
    savefig(fig, 'kmeans_barrido_k')


def plot_gmm_bic(gmm_df, best_n):
    fig, ax = plt.subplots(figsize=(8, 4), dpi=300)
    fig.patch.set_facecolor('#F7F9FC')
    ax.plot(gmm_df['n_components'], gmm_df['BIC'], marker='o', color='#1E88E5', label='BIC')
    ax.plot(gmm_df['n_components'], gmm_df['AIC'], marker='s', color='#F4511E', label='AIC')
    ax.axvline(N_CLASSES, color='#999999', linestyle=':', linewidth=1,
               label=f'nº de agarres = {N_CLASSES}')
    ax.axvline(best_n, color='#2E7D32', linestyle='--', linewidth=1.2,
               label=f'mejor BIC = {best_n}')
    ax.set_xlabel('n_components (número de grupos)', fontweight='bold')
    ax.set_ylabel('Criterio (menor = mejor)', fontweight='bold')
    ax.set_title('Selección del número de grupos con GMM', fontweight='bold')
    ax.legend(fontsize=8)
    style_ax(ax)
    plt.tight_layout()
    savefig(fig, 'gmm_bic_aic')


def plot_agg_linkages(agg_df):
    fig, ax = plt.subplots(figsize=(7, 4), dpi=300)
    fig.patch.set_facecolor('#F7F9FC')
    pos, width = np.arange(len(agg_df)), 0.27
    ax.bar(pos - width, agg_df['ARI_agarre'], width, label='ARI', color='#F4511E')
    ax.bar(pos, agg_df['NMI_agarre'], width, label='NMI', color='#43A047')
    ax.bar(pos + width, agg_df['silhouette'], width, label='Silhouette', color='#1E88E5')
    ax.set_xticks(pos, agg_df.index)
    ax.set_title(f'Aglomerativo con k={N_CLASSES} según linkage', fontweight='bold')
    ax.set_ylabel('Valor', fontweight='bold')
    ax.legend(fontsize=8, frameon=True)
    style_ax(ax)
    plt.tight_layout()
    savefig(fig, 'aglomerativo_linkages')


def plot_azar(nulls, p_vals, tests, fname='azar_histogramas'):
    fig, axes = plt.subplots(1, len(tests), figsize=(4.6 * len(tests), 3.6), dpi=300)
    fig.patch.set_facecolor('#F7F9FC')
    for ax, t in zip(np.atleast_1d(axes), tests):
        null = nulls[t['nombre']]
        p = p_vals[t['nombre']]
        style_ax(ax)
        ax.hist(null, bins=40, color='#B0BEC5', edgecolor='white',
                label='ARI con etiquetas al azar')
        ax.axvline(t['ari'], color=t['color'], linewidth=1.6,
                   label=f"ARI real = {t['ari']:.3f}")
        ax.set_xlabel('ARI', fontweight='bold')
        ax.set_ylabel('Frecuencia', fontweight='bold')
        ax.set_title(f"{t['nombre']}\np = {p:.4f}", fontweight='bold', fontsize=10)
        ax.legend(fontsize=7)
    plt.tight_layout()
    savefig(fig, fname)


# Coordenadas 2D y 3D: se calculan una sola vez y se usan en todas las vistas
E2D, AXIS = proyectar(X, 2)
E3D, _ = proyectar(X, 3)

## 2. K-Means con k = número de agarres

Se le pide a K-Means que forme tantos grupos como agarres hay.

**Qué mirar:**
- **ARI / NMI agarre:** cuánto coinciden los grupos con los agarres (0 = como al azar, 1 = perfecto).
- **ARI sujeto:** control. Si es mayor que el de agarre, los grupos separan personas, no agarres.
- **Grupos por sujeto:** en cuántos grupos distintos caen los agarres de una misma persona. Cerca del
  número de agarres = bien; cerca de 1 = agrupa por persona.
- **Tabla:** filas = grupos encontrados, columnas = agarres reales; el número es la **proporción (%)**
  de cada agarre que cayó en ese grupo (cada columna suma 100). Los conteos crudos quedan en
  `<metodo>_tabla_conteos.csv`.
- **Vistas 2D y 3D:** izquierda coloreada por grupo, derecha por agarre real, con las mismas
  coordenadas. La proyección es solo para ver; las conclusiones salen del ARI y de la tabla.

In [ ]:
km_labels, km_res = run_kmeans(X, y, groups, k=N_CLASSES)
display(pd.Series(km_res).round(3).to_frame('K-Means'))

km_title = (f'K-Means k={N_CLASSES} — ARI={km_res["ARI_agarre"]:.3f} | '
            f'NMI={km_res["NMI_agarre"]:.3f}')
plot_contingency(km_labels, km_title, 'kmeans_tabla')
plot_clusters_2d(km_labels, km_title, 'kmeans_2d')
plot_clusters_3d(km_labels, km_title, 'kmeans_3d')

## 3. K-Means con otros valores de k

Se repite K-Means con distintos números de grupos.
- **Silhouette** (izquierda): qué tan bien separados quedan los grupos. El k con el valor más alto es el
  número de grupos que "prefieren" los datos. Si es menor que el número de agarres, algunos agarres forman
  familias.
- **ARI / NMI** (derecha): coincidencia con los agarres para cada k.

In [ ]:
sweep_df, best_k = barrido_kmeans(X, y)
plot_barrido_k(sweep_df, best_k)
print(f'k con mejor separación (silhouette): {best_k}  |  número de agarres: {N_CLASSES}')

## 4. GMM (Mezcla de Gaussianas)

A diferencia de K-Means, GMM modela cada grupo como una **elipse** (una gaussiana con su matriz
de covarianza) y asigna probabilidades en vez de distancias duras.

Se prueba `n_components` de 2 a 12 y se elige el mejor con **BIC** (menor = mejor; penaliza
modelos con más grupos para evitar sobreajustar). La línea punteada marca el número real de agarres.

**Qué mirar:** si el mínimo del BIC cae en 7, los datos "piden" 7 grupos; si cae en un valor
menor, algunos agarres forman familias indistinguibles.

In [ ]:
gmm_df, best_n, gmm_labels, gmm_res = run_gmm(X, y, groups)
plot_gmm_bic(gmm_df, best_n)
print(f'Mejor n_components según BIC: {best_n}  |  número de agarres: {N_CLASSES}')

display(pd.Series(gmm_res).round(3).to_frame('GMM'))
gmm_title = (f'GMM n={best_n} — ARI={gmm_res["ARI_agarre"]:.3f} | '
             f'NMI={gmm_res["NMI_agarre"]:.3f}')
plot_contingency(gmm_labels, gmm_title, 'gmm_tabla')
plot_clusters_2d(gmm_labels, gmm_title, 'gmm_2d')
plot_clusters_3d(gmm_labels, gmm_title, 'gmm_3d')

## 5. Aglomerativo (jerárquico)

Construye un árbol de grupos fusionando los pares más cercanos y lo corta en `k = nº de agarres`.
Se comparan tres formas de medir la distancia entre grupos (**linkages**):

- **ward:** minimiza la varianza interna de los grupos (el más parecido a K-Means).
- **complete:** usa la distancia entre los puntos más lejanos de cada grupo.
- **average:** usa la distancia promedio.

**Qué mirar:** si los tres linkages dan ARI parecido, la estructura es robusta al método;
si difieren mucho, la decisión de agrupar depende del criterio.

In [ ]:
agg_df, agg_labels, best_link = run_aglomerativo(X, y, groups, k=N_CLASSES)
agg_best = agg_labels[best_link]
agg_res = agg_df.loc[best_link]
display(agg_df.round(3))
plot_agg_linkages(agg_df)

agg_title = (f'Aglomerativo ({best_link}, k={N_CLASSES}) — ARI={agg_res.ARI_agarre:.3f} | '
             f'NMI={agg_res.NMI_agarre:.3f}')
plot_contingency(agg_best, agg_title, 'aglomerativo_tabla')
plot_clusters_2d(agg_best, agg_title, 'aglomerativo_2d')
plot_clusters_3d(agg_best, agg_title, 'aglomerativo_3d')

## 6. HDBSCAN con min_samples explícito

HDBSCAN decide solo cuántos grupos hay y marca como **ruido (−1)** lo que no encaja en ninguno.

**Por qué la versión anterior daba 0 grupos y 100% ruido:**

1. **`min_samples` por defecto en sklearn = `min_cluster_size`.** Con `min_cluster_size` de 15 a 40,
   cada punto necesitaba 15–40 vecinos en su radio de densidad para ser "núcleo". Con 609 filas en
   8 dimensiones eso es tan exigente que **ningún punto califica → todo es ruido → 0 grupos**.
2. **El dataset no tiene islas de densidad.** HDBSCAN busca regiones densas separadas por zonas
   raras; aquí la densidad es casi uniforme y los agarres se solapan, así que incluso bien ajustado
   encuentra pocos grupos con ARI bajo. Eso también es un resultado.

Aquí se barre `min_cluster_size` × `min_samples` con `min_samples` pequeño (1–10).
Un ARI alto con mucho ruido no es un buen resultado: hay que mirar ambos.

In [ ]:
hdb_df, hdb_labels, best_hdb = run_hdbscan(X, y, groups)
display(hdb_df.round(3))

if best_hdb is not None:
    r = hdb_df.loc[best_hdb]
    hdb_title = (f'HDBSCAN (mcs={best_hdb[0]}, min_samples={best_hdb[1]}) — '
                 f'{int(r.n_grupos)} grupos, {r["%_ruido"]:.0f}% ruido, ARI={r.ARI_agarre:.3f}')
    plot_contingency(hdb_labels[best_hdb], hdb_title, 'hdbscan_tabla')
    plot_clusters_2d(hdb_labels[best_hdb], hdb_title, 'hdbscan_2d')
    plot_clusters_3d(hdb_labels[best_hdb], hdb_title, 'hdbscan_3d')
else:
    print('HDBSCAN no encontró más de un grupo con ninguna combinación '
          '(todo es un solo grupo o ruido): no hay regiones densas separadas en este dataset.')

## 7. ¿Es mejor que el azar?

Se mezclan al azar las etiquetas de agarre `N_PERM` veces y se calcula el ARI de cada algoritmo
frente a su propia distribución de azar. Si el ARI real queda muy a la derecha del histograma
(p-valor bajo), la coincidencia no es casualidad.

In [ ]:
tests = [
    {'nombre': 'K-Means', 'labels': km_labels, 'ari': km_res['ARI_agarre'], 'color': '#E53935'},
    {'nombre': 'GMM', 'labels': gmm_labels, 'ari': gmm_res['ARI_agarre'], 'color': '#1E88E5'},
    {'nombre': f'Aglomerativo ({best_link})', 'labels': agg_best,
     'ari': float(agg_res.ARI_agarre), 'color': '#43A047'},
]
nulls, p_vals = prueba_azar(y, tests)
plot_azar(nulls, p_vals, tests)

## 8. Resumen y guardado

Se guarda un `resumen.csv` por dataset con las métricas de **todos** los algoritmos, más los
barridos individuales y las asignaciones por fila. Para comparar con otro dataset, se ejecuta
este notebook con ese dataset y se comparan los dos `resumen.csv`.

In [ ]:
etiquetas = {'kmeans': km_labels, 'gmm': gmm_labels,
             f'aglomerativo_{best_link}': agg_best}
if best_hdb is not None:
    etiquetas['hdbscan'] = hdb_labels[best_hdb]
assign = armar_asignaciones(df, etiquetas)

resumen = {
    'dataset': DATASET_TAG,
    'n_columnas': N_FEATURES,
    'kmeans_ARI_agarre': km_res['ARI_agarre'],
    'kmeans_NMI_agarre': km_res['NMI_agarre'],
    'kmeans_ARI_sujeto': km_res['ARI_sujeto'],
    'kmeans_grupos_por_sujeto': km_res['grupos_por_sujeto'],
    'kmeans_p_azar': p_vals['K-Means'],
    'k_mejor_separacion': best_k,
    'gmm_n_bic': best_n,
    'gmm_ARI_agarre': gmm_res['ARI_agarre'],
    'gmm_NMI_agarre': gmm_res['NMI_agarre'],
    'gmm_p_azar': p_vals['GMM'],
    'agg_linkage': best_link,
    'agg_ARI_agarre': float(agg_res.ARI_agarre),
    'agg_NMI_agarre': float(agg_res.NMI_agarre),
    'agg_p_azar': p_vals[f'Aglomerativo ({best_link})'],
    'hdbscan_n_grupos': None if best_hdb is None else int(hdb_df.loc[best_hdb, 'n_grupos']),
    'hdbscan_pct_ruido': None if best_hdb is None else hdb_df.loc[best_hdb, '%_ruido'],
    'hdbscan_ARI_agarre': None if best_hdb is None else hdb_df.loc[best_hdb, 'ARI_agarre'],
}
guardar_resultados(RESULTS_DIR, resumen, assign, {
    'kmeans_barrido_k': sweep_df,
    'gmm_barrido_n': gmm_df,
    'aglomerativo_linkages': agg_df,
    'hdbscan_metricas': hdb_df,
})
display(pd.Series(resumen).to_frame('valor'))

## 9. Interpretación

*(Completar con los resultados.)*

- **¿Se forman los agarres solos?** → ARI/NMI de K-Means, GMM y aglomerativo; p-valores frente al azar.
- **¿Qué agarres se mezclan?** → tabla de K-Means/GMM; comparar con los pares más confundidos de la LDA.
- **¿Se agrupa por persona?** → ARI sujeto y grupos por sujeto.
- **¿Cuántos grupos sugieren los datos?** → k con mejor separación, mínimo del BIC de GMM y
  resultado de HDBSCAN (con `min_samples` explícito).
- **¿Hay densidades separadas?** → si HDBSCAN da ruido alto con ARI bajo, los datos no forman
  islas densas: es una propiedad del dataset, no un bug del algoritmo.

In [ ]:
sello.guardar()